# Week 13: CCE-Triggered Retrieval Intervention Study

## The Real Hypothesis

**When the model is confused (high CCE), it gives wrong answers. Providing relevant context at that moment improves accuracy.**

## Why Previous Experiments Missed This

- Week13_V2: Only measured correlation (CCE vs error) - didn't test intervention
- Week13_CodeGen: Same issue - correlation without intervention

## Correct Experiment Design

For each difficult question:

| Condition | Method | Expected |
|-----------|--------|----------|
| A: No retrieval | Generate directly | Lower accuracy |
| B: CCE-triggered | Retrieve when CCE > threshold | Higher accuracy |
| C: Always retrieve | Retrieve before every answer | High accuracy but inefficient |
| D: Random retrieval | Retrieve at random points | Moderate accuracy |

**Key Metric:** Does B (CCE-triggered) achieve similar accuracy to C (always) with fewer retrieval calls?

## What "Retrieval" Means Here

When CCE spikes during generation:
1. Pause generation
2. Use the partial output to retrieve relevant source code
3. Inject the retrieved context
4. Continue generation with the new context

In [ ]:
# Install dependencies
!pip install -q torch transformers accelerate bitsandbytes sentence-transformers scipy numpy pandas matplotlib

In [ ]:
import os
import re
import json
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import random
from typing import List, Dict, Tuple, Optional, Callable
from dataclasses import dataclass, field
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer
import matplotlib.pyplot as plt
from scipy import stats

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

In [ ]:
# Clone httpx for ground truth
REPO_URL = 'https://github.com/encode/httpx.git'
REPO_DIR = '/content/httpx'

if not os.path.exists(REPO_DIR):
    !git clone --depth 1 {REPO_URL} {REPO_DIR}

def load_source_files(repo_dir: str, src_folder: str = 'httpx') -> Dict[str, str]:
    files = {}
    src_path = os.path.join(repo_dir, src_folder)
    for root, _, filenames in os.walk(src_path):
        for fname in filenames:
            if fname.endswith('.py'):
                fpath = os.path.join(root, fname)
                rel_path = os.path.relpath(fpath, repo_dir)
                with open(fpath, 'r', encoding='utf-8') as f:
                    files[rel_path] = f.read()
    return files

SOURCE_FILES = load_source_files(REPO_DIR)
print(f"Loaded {len(SOURCE_FILES)} source files")

In [ ]:
# Load models
MODEL_NAME = "codellama/CodeLlama-7b-Instruct-hf"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Embedding model for retrieval
embedder = SentenceTransformer('all-MiniLM-L6-v2')

print("Models loaded!")

## Build Retrieval Index

In [ ]:
# Create chunks from source files
def chunk_source_files(source_files: Dict[str, str], chunk_size: int = 50) -> List[Dict]:
    """Split source files into chunks for retrieval."""
    chunks = []
    
    for filepath, content in source_files.items():
        lines = content.split('\n')
        
        for i in range(0, len(lines), chunk_size // 2):  # 50% overlap
            chunk_lines = lines[i:i + chunk_size]
            if len(chunk_lines) < 10:  # Skip tiny chunks
                continue
            
            chunk_text = '\n'.join(chunk_lines)
            chunks.append({
                'filepath': filepath,
                'start_line': i + 1,
                'end_line': i + len(chunk_lines),
                'content': chunk_text,
            })
    
    return chunks

SOURCE_CHUNKS = chunk_source_files(SOURCE_FILES)
print(f"Created {len(SOURCE_CHUNKS)} chunks")

# Embed all chunks
print("Embedding chunks...")
chunk_texts = [c['content'] for c in SOURCE_CHUNKS]
CHUNK_EMBEDDINGS = embedder.encode(chunk_texts, show_progress_bar=True, convert_to_tensor=True)
print(f"Embeddings shape: {CHUNK_EMBEDDINGS.shape}")

In [ ]:
def retrieve_context(query: str, top_k: int = 3) -> str:
    """Retrieve relevant source code chunks for a query."""
    query_embedding = embedder.encode(query, convert_to_tensor=True)
    
    # Compute similarities
    similarities = torch.nn.functional.cosine_similarity(
        query_embedding.unsqueeze(0), CHUNK_EMBEDDINGS
    )
    
    # Get top-k
    top_indices = torch.topk(similarities, k=min(top_k, len(SOURCE_CHUNKS))).indices
    
    # Build context string
    context_parts = []
    for idx in top_indices:
        chunk = SOURCE_CHUNKS[idx.item()]
        context_parts.append(
            f"# From {chunk['filepath']} (lines {chunk['start_line']}-{chunk['end_line']}):\n"
            f"{chunk['content'][:1000]}"  # Limit chunk size
        )
    
    return "\n\n".join(context_parts)

# Test retrieval
test_query = "How does httpx handle timeouts?"
test_context = retrieve_context(test_query)
print(f"Query: {test_query}")
print(f"\nRetrieved context (first 500 chars):")
print(test_context[:500])

## Difficult Questions

Questions where the model is likely to be uncertain or wrong without context.

In [ ]:
# Difficult questions that require specific knowledge
DIFFICULT_QUESTIONS = [
    # Implementation details (model likely doesn't know)
    {
        'id': 'q01',
        'question': 'What is the exact default timeout value in seconds that httpx uses?',
        'verify': lambda r: '5' in r and ('second' in r.lower() or 'timeout' in r.lower()),
        'relevant_file': 'httpx/_config.py',
    },
    {
        'id': 'q02', 
        'question': 'What is the default max_redirects value in httpx?',
        'verify': lambda r: '20' in r,
        'relevant_file': 'httpx/_config.py',
    },
    {
        'id': 'q03',
        'question': 'What compression formats does httpx automatically decode? List all of them.',
        'verify': lambda r: 'gzip' in r.lower() and ('brotli' in r.lower() or 'deflate' in r.lower()),
        'relevant_file': 'httpx/_decoders.py',
    },
    {
        'id': 'q04',
        'question': 'What is the name of the base exception class for all httpx exceptions?',
        'verify': lambda r: 'HTTPError' in r,
        'relevant_file': 'httpx/_exceptions.py',
    },
    {
        'id': 'q05',
        'question': 'What authentication classes does httpx provide? Name all of them.',
        'verify': lambda r: 'BasicAuth' in r and 'DigestAuth' in r,
        'relevant_file': 'httpx/_auth.py',
    },
    
    # Tricky behavior questions
    {
        'id': 'q06',
        'question': 'Does httpx follow redirects by default? What is the exact default behavior?',
        'verify': lambda r: 'false' in r.lower() or 'no' in r.lower() or 'not' in r.lower(),
        'relevant_file': 'httpx/_client.py',
    },
    {
        'id': 'q07',
        'question': 'What happens when you call response.json() on a non-JSON response in httpx?',
        'verify': lambda r: 'error' in r.lower() or 'exception' in r.lower() or 'raise' in r.lower(),
        'relevant_file': 'httpx/_models.py',
    },
    {
        'id': 'q08',
        'question': 'What is the difference between httpx.TimeoutException and httpx.ConnectTimeout?',
        'verify': lambda r: 'connect' in r.lower() or 'base' in r.lower() or 'parent' in r.lower(),
        'relevant_file': 'httpx/_exceptions.py',
    },
    
    # Code generation questions
    {
        'id': 'q09',
        'question': 'Write code to create an httpx Client with a 10 second timeout and base_url.',
        'verify': lambda r: 'Client' in r and 'timeout' in r.lower() and 'base_url' in r.lower(),
        'relevant_file': 'httpx/_client.py',
    },
    {
        'id': 'q10',
        'question': 'Write code to catch and handle a timeout exception in httpx.',
        'verify': lambda r: 'try' in r.lower() and 'except' in r.lower() and 'timeout' in r.lower(),
        'relevant_file': 'httpx/_exceptions.py',
    },
    
    # Specific API details
    {
        'id': 'q11',
        'question': 'What parameters does the httpx.Timeout class accept?',
        'verify': lambda r: 'connect' in r.lower() or 'read' in r.lower() or 'write' in r.lower(),
        'relevant_file': 'httpx/_config.py',
    },
    {
        'id': 'q12',
        'question': 'How do you configure httpx to use HTTP/2?',
        'verify': lambda r: 'http2' in r.lower() and 'true' in r.lower(),
        'relevant_file': 'httpx/_client.py',
    },
    {
        'id': 'q13',
        'question': 'What method checks if an httpx response status code indicates success?',
        'verify': lambda r: 'is_success' in r or 'is_error' in r,
        'relevant_file': 'httpx/_models.py',
    },
    {
        'id': 'q14',
        'question': 'How do you stream response content in httpx without loading it all into memory?',
        'verify': lambda r: 'iter_bytes' in r or 'iter_text' in r or 'stream' in r.lower(),
        'relevant_file': 'httpx/_models.py',
    },
    {
        'id': 'q15',
        'question': 'What is the httpx equivalent of requests.Session for connection pooling?',
        'verify': lambda r: 'Client' in r,
        'relevant_file': 'httpx/_client.py',
    },
    
    # Edge cases
    {
        'id': 'q16',
        'question': 'What exception does httpx raise for SSL certificate verification failures?',
        'verify': lambda r: 'ConnectError' in r or 'SSL' in r or 'certificate' in r.lower(),
        'relevant_file': 'httpx/_exceptions.py',
    },
    {
        'id': 'q17',
        'question': 'How do you send a request with both query parameters and a JSON body in httpx?',
        'verify': lambda r: 'params' in r.lower() and 'json' in r.lower(),
        'relevant_file': 'httpx/_client.py',
    },
    {
        'id': 'q18',
        'question': 'What is the default User-Agent string format used by httpx?',
        'verify': lambda r: 'python-httpx' in r.lower() or 'httpx/' in r.lower(),
        'relevant_file': 'httpx/_client.py',
    },
    {
        'id': 'q19',
        'question': 'How does httpx handle cookies across multiple requests?',
        'verify': lambda r: 'client' in r.lower() or 'cookies' in r.lower() or 'jar' in r.lower(),
        'relevant_file': 'httpx/_client.py',
    },
    {
        'id': 'q20',
        'question': 'What transport classes does httpx use internally?',
        'verify': lambda r: 'HTTPTransport' in r or 'transport' in r.lower(),
        'relevant_file': 'httpx/_transports/',
    },
]

print(f"Defined {len(DIFFICULT_QUESTIONS)} difficult questions")

## Generation with CCE-Triggered Retrieval

In [ ]:
@dataclass
class GenerationResult:
    question_id: str
    question: str
    condition: str  # 'no_retrieval', 'cce_triggered', 'always_retrieve', 'random'
    response: str
    is_correct: bool
    cce_values: List[float]
    retrieval_count: int = 0
    retrieval_positions: List[int] = field(default_factory=list)
    
    @property
    def mean_cce(self) -> float:
        return np.mean(self.cce_values) if self.cce_values else 0.0
    
    @property
    def max_cce(self) -> float:
        return np.max(self.cce_values) if self.cce_values else 0.0

In [ ]:
def generate_with_retrieval(
    model, tokenizer, question: str,
    retrieval_mode: str = 'none',  # 'none', 'cce_triggered', 'always', 'random'
    cce_threshold: float = 2.0,
    max_tokens: int = 150,
    context: str = None,  # Pre-provided context for 'always' mode
) -> Tuple[str, List[float], int, List[int]]:
    """Generate response with optional CCE-triggered retrieval.
    
    Returns: (response, cce_values, retrieval_count, retrieval_positions)
    """
    
    # Build initial prompt
    if retrieval_mode == 'always' and context:
        prompt = f"""[INST] Use the following source code reference to answer the question.

Source Code:
{context[:2000]}

Question: {question}

Answer concisely: [/INST]"""
    else:
        prompt = f"""[INST] Answer this question about the httpx Python library concisely.

Question: {question} [/INST]"""
    
    input_ids = tokenizer.encode(prompt, return_tensors='pt').to(model.device)
    
    cce_values = []
    generated_tokens = []
    retrieval_count = 0
    retrieval_positions = []
    
    # For random mode, decide retrieval points upfront
    random_retrieval_at = set()
    if retrieval_mode == 'random':
        # Retrieve at ~20% of positions
        num_retrievals = max(1, max_tokens // 5)
        random_retrieval_at = set(random.sample(range(10, max_tokens), min(num_retrievals, max_tokens - 10)))
    
    with torch.no_grad():
        for step in range(max_tokens):
            outputs = model(input_ids)
            logits = outputs.logits[:, -1, :].float()
            
            # Compute CCE (no temperature)
            probs = F.softmax(logits, dim=-1)
            log2_probs = torch.log2(probs + 1e-10)
            entropy = -(probs * log2_probs).sum(dim=-1).item()
            cce_values.append(entropy)
            
            # Check if we should retrieve
            should_retrieve = False
            
            if retrieval_mode == 'cce_triggered' and entropy > cce_threshold and retrieval_count == 0:
                # Only retrieve once when CCE first exceeds threshold
                should_retrieve = True
            elif retrieval_mode == 'random' and step in random_retrieval_at and retrieval_count == 0:
                should_retrieve = True
            
            if should_retrieve:
                # Get partial response so far
                partial_response = tokenizer.decode(generated_tokens, skip_special_tokens=True)
                
                # Retrieve relevant context
                query = f"{question} {partial_response}"
                retrieved_context = retrieve_context(query, top_k=2)
                
                # Rebuild prompt with context
                new_prompt = f"""[INST] Use this source code to answer the question.

Source Code:
{retrieved_context[:1500]}

Question: {question}

Partial answer so far: {partial_response}

Continue and complete the answer: [/INST]"""
                
                input_ids = tokenizer.encode(new_prompt, return_tensors='pt').to(model.device)
                retrieval_count += 1
                retrieval_positions.append(step)
                
                # Re-run to get new logits after context injection
                outputs = model(input_ids)
                logits = outputs.logits[:, -1, :].float()
            
            # Sample next token (greedy)
            next_token = torch.argmax(logits, dim=-1, keepdim=True)
            generated_tokens.append(next_token.item())
            input_ids = torch.cat([input_ids, next_token], dim=-1)
            
            if next_token.item() == tokenizer.eos_token_id:
                break
    
    response = tokenizer.decode(generated_tokens, skip_special_tokens=True)
    return response, cce_values, retrieval_count, retrieval_positions

In [ ]:
def run_condition(questions: List[Dict], condition: str, cce_threshold: float = 2.0) -> List[GenerationResult]:
    """Run all questions under a specific condition."""
    results = []
    
    for i, q in enumerate(questions):
        # Pre-retrieve context for 'always' mode
        context = None
        if condition == 'always_retrieve':
            context = retrieve_context(q['question'], top_k=3)
        
        # Map condition name to mode
        mode_map = {
            'no_retrieval': 'none',
            'cce_triggered': 'cce_triggered',
            'always_retrieve': 'always',
            'random': 'random',
        }
        
        response, cce_vals, ret_count, ret_pos = generate_with_retrieval(
            model, tokenizer, q['question'],
            retrieval_mode=mode_map[condition],
            cce_threshold=cce_threshold,
            context=context,
        )
        
        is_correct = q['verify'](response)
        
        result = GenerationResult(
            question_id=q['id'],
            question=q['question'],
            condition=condition,
            response=response,
            is_correct=is_correct,
            cce_values=cce_vals,
            retrieval_count=ret_count,
            retrieval_positions=ret_pos,
        )
        results.append(result)
        
        status = 'OK' if is_correct else 'ERR'
        ret_info = f"(retrieved at {ret_pos})" if ret_count > 0 else ""
        
        if (i + 1) % 5 == 0:
            acc = sum(1 for r in results if r.is_correct) / len(results)
            print(f"  [{i+1}/{len(questions)}] Accuracy: {acc:.1%}")
    
    return results

## Run Experiment: All Conditions

In [ ]:
%%time

CONDITIONS = ['no_retrieval', 'cce_triggered', 'always_retrieve', 'random']
CCE_THRESHOLD = 2.0  # Threshold for triggering retrieval

all_results = {}

for condition in CONDITIONS:
    print(f"\n{'='*60}")
    print(f"CONDITION: {condition}")
    print(f"{'='*60}")
    
    results = run_condition(DIFFICULT_QUESTIONS, condition, cce_threshold=CCE_THRESHOLD)
    all_results[condition] = results
    
    # Summary
    accuracy = sum(1 for r in results if r.is_correct) / len(results)
    total_retrievals = sum(r.retrieval_count for r in results)
    mean_cce = np.mean([r.mean_cce for r in results])
    
    print(f"\nResults for {condition}:")
    print(f"  Accuracy: {accuracy:.1%} ({sum(1 for r in results if r.is_correct)}/{len(results)})")
    print(f"  Total retrievals: {total_retrievals}")
    print(f"  Mean CCE: {mean_cce:.3f}")

## Analysis: Compare Conditions

In [ ]:
print("="*70)
print("COMPARISON OF CONDITIONS")
print("="*70)

comparison_data = []

for condition, results in all_results.items():
    accuracy = sum(1 for r in results if r.is_correct) / len(results)
    total_retrievals = sum(r.retrieval_count for r in results)
    mean_cce = np.mean([r.mean_cce for r in results])
    
    # Efficiency: accuracy per retrieval call
    if condition == 'always_retrieve':
        # Always retrieve counts as 1 retrieval per question
        efficiency = accuracy / len(results) if len(results) > 0 else 0
        total_retrievals = len(results)  # One retrieval per question
    elif total_retrievals > 0:
        efficiency = accuracy / total_retrievals
    else:
        efficiency = accuracy  # No retrievals, efficiency = accuracy
    
    comparison_data.append({
        'condition': condition,
        'accuracy': accuracy,
        'total_retrievals': total_retrievals,
        'mean_cce': mean_cce,
        'efficiency': efficiency,
    })

df_comparison = pd.DataFrame(comparison_data)
print(df_comparison.to_string(index=False))

In [ ]:
# Statistical comparison: CCE-triggered vs No Retrieval
print("\n" + "="*70)
print("STATISTICAL ANALYSIS")
print("="*70)

# Compare accuracy between conditions
no_ret = all_results['no_retrieval']
cce_ret = all_results['cce_triggered']
always_ret = all_results['always_retrieve']

# Paired comparison: same questions, different conditions
no_ret_correct = [1 if r.is_correct else 0 for r in no_ret]
cce_ret_correct = [1 if r.is_correct else 0 for r in cce_ret]
always_ret_correct = [1 if r.is_correct else 0 for r in always_ret]

# McNemar's test or simple proportion comparison
from scipy.stats import chi2_contingency

def compare_conditions(cond1_correct, cond2_correct, name1, name2):
    """Compare two conditions using McNemar-like analysis."""
    # Count: both correct, both wrong, only cond1 correct, only cond2 correct
    both_correct = sum(1 for a, b in zip(cond1_correct, cond2_correct) if a == 1 and b == 1)
    both_wrong = sum(1 for a, b in zip(cond1_correct, cond2_correct) if a == 0 and b == 0)
    only_1_correct = sum(1 for a, b in zip(cond1_correct, cond2_correct) if a == 1 and b == 0)
    only_2_correct = sum(1 for a, b in zip(cond1_correct, cond2_correct) if a == 0 and b == 1)
    
    print(f"\n{name1} vs {name2}:")
    print(f"  Both correct: {both_correct}")
    print(f"  Both wrong: {both_wrong}")
    print(f"  Only {name1} correct: {only_1_correct}")
    print(f"  Only {name2} correct: {only_2_correct}")
    
    # Improvement
    improvement = only_2_correct - only_1_correct
    print(f"  Net improvement ({name2} over {name1}): {improvement:+d} questions")
    
    return improvement

improve_cce = compare_conditions(no_ret_correct, cce_ret_correct, 'no_retrieval', 'cce_triggered')
improve_always = compare_conditions(no_ret_correct, always_ret_correct, 'no_retrieval', 'always_retrieve')
improve_cce_vs_always = compare_conditions(cce_ret_correct, always_ret_correct, 'cce_triggered', 'always_retrieve')

In [ ]:
# Visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Accuracy by condition
ax1 = axes[0, 0]
conditions = df_comparison['condition'].tolist()
accuracies = df_comparison['accuracy'].tolist()
colors = ['red', 'green', 'blue', 'orange']
bars = ax1.bar(conditions, accuracies, color=colors, alpha=0.7)
ax1.set_ylabel('Accuracy')
ax1.set_title('Accuracy by Retrieval Condition')
ax1.set_ylim(0, 1)
for bar, acc in zip(bars, accuracies):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
             f'{acc:.1%}', ha='center', va='bottom')

# 2. Retrieval count by condition
ax2 = axes[0, 1]
retrievals = df_comparison['total_retrievals'].tolist()
ax2.bar(conditions, retrievals, color=colors, alpha=0.7)
ax2.set_ylabel('Total Retrieval Calls')
ax2.set_title('Retrieval Calls by Condition')

# 3. CCE distribution when retrieval was triggered
ax3 = axes[1, 0]
cce_at_retrieval = []
for r in all_results['cce_triggered']:
    for pos in r.retrieval_positions:
        if pos < len(r.cce_values):
            cce_at_retrieval.append(r.cce_values[pos])

if cce_at_retrieval:
    ax3.hist(cce_at_retrieval, bins=15, color='green', alpha=0.7)
    ax3.axvline(x=CCE_THRESHOLD, color='red', linestyle='--', label=f'Threshold={CCE_THRESHOLD}')
    ax3.set_xlabel('CCE at Retrieval Point')
    ax3.set_ylabel('Count')
    ax3.set_title('CCE Values When Retrieval Was Triggered')
    ax3.legend()
else:
    ax3.text(0.5, 0.5, 'No retrievals triggered', ha='center', va='center')

# 4. Per-question comparison
ax4 = axes[1, 1]
question_ids = [r.question_id for r in no_ret]
x = np.arange(len(question_ids))
width = 0.25

ax4.bar(x - width, no_ret_correct, width, label='No Retrieval', alpha=0.7)
ax4.bar(x, cce_ret_correct, width, label='CCE-Triggered', alpha=0.7)
ax4.bar(x + width, always_ret_correct, width, label='Always Retrieve', alpha=0.7)
ax4.set_xlabel('Question')
ax4.set_ylabel('Correct (1) / Wrong (0)')
ax4.set_title('Per-Question Results')
ax4.set_xticks(x)
ax4.set_xticklabels([q['id'] for q in DIFFICULT_QUESTIONS], rotation=45)
ax4.legend()

plt.tight_layout()
plt.savefig('week13_intervention_results.png', dpi=150)
plt.show()

In [ ]:
# Final conclusion
print("="*70)
print("FINAL CONCLUSION: CCE-Triggered Retrieval Intervention")
print("="*70)

no_ret_acc = sum(no_ret_correct) / len(no_ret_correct)
cce_ret_acc = sum(cce_ret_correct) / len(cce_ret_correct)
always_ret_acc = sum(always_ret_correct) / len(always_ret_correct)

cce_retrievals = sum(r.retrieval_count for r in cce_ret)
always_retrievals = len(always_ret)  # One per question

print(f"""
RESULTS SUMMARY:

| Condition        | Accuracy | Retrievals | Efficiency |
|------------------|----------|------------|------------|
| No Retrieval     | {no_ret_acc:.1%}    | 0          | baseline   |
| CCE-Triggered    | {cce_ret_acc:.1%}    | {cce_retrievals}          | {cce_ret_acc/max(1,cce_retrievals):.2f}/call   |
| Always Retrieve  | {always_ret_acc:.1%}    | {always_retrievals}         | {always_ret_acc/always_retrievals:.2f}/call   |

KEY FINDINGS:
""")

# Analyze results
if cce_ret_acc > no_ret_acc:
    improvement = (cce_ret_acc - no_ret_acc) / no_ret_acc * 100
    print(f"1. CCE-triggered retrieval IMPROVES accuracy by {improvement:.1f}% over no retrieval")
else:
    print(f"1. CCE-triggered retrieval does NOT improve accuracy over no retrieval")

if cce_ret_acc >= always_ret_acc * 0.9:  # Within 10% of always
    if cce_retrievals < always_retrievals:
        print(f"2. CCE-triggered achieves similar accuracy to always-retrieve with {always_retrievals - cce_retrievals} FEWER retrieval calls")
    else:
        print(f"2. CCE-triggered matches always-retrieve accuracy")
else:
    print(f"2. CCE-triggered is less accurate than always-retrieve")

if cce_retrievals > 0:
    print(f"3. CCE threshold {CCE_THRESHOLD} triggered {cce_retrievals} retrievals across {len(DIFFICULT_QUESTIONS)} questions")
else:
    print(f"3. CCE threshold {CCE_THRESHOLD} may be too high - no retrievals triggered")

print(f"""
HYPOTHESIS EVALUATION:
"When the model is confused (high CCE), retrieval improves accuracy."
""")

if cce_ret_acc > no_ret_acc and cce_retrievals > 0:
    print("SUPPORTED: CCE-triggered retrieval improves accuracy when model shows confusion.")
elif cce_retrievals == 0:
    print("INCONCLUSIVE: CCE threshold too high, need to lower threshold and re-run.")
else:
    print("NOT SUPPORTED: Retrieval at CCE spikes did not improve accuracy.")

In [ ]:
# Save results
output = {
    'experiment': 'Week13_Retrieval_Intervention',
    'cce_threshold': CCE_THRESHOLD,
    'num_questions': len(DIFFICULT_QUESTIONS),
    'conditions': {},
}

for condition, results in all_results.items():
    output['conditions'][condition] = {
        'accuracy': sum(1 for r in results if r.is_correct) / len(results),
        'total_retrievals': sum(r.retrieval_count for r in results),
        'per_question': [
            {
                'question_id': r.question_id,
                'is_correct': r.is_correct,
                'retrieval_count': r.retrieval_count,
                'retrieval_positions': r.retrieval_positions,
                'mean_cce': float(r.mean_cce),
                'max_cce': float(r.max_cce),
            }
            for r in results
        ],
    }

with open('week13_intervention_results.json', 'w') as f:
    json.dump(output, f, indent=2)

print("Results saved to week13_intervention_results.json")

## Threshold Sensitivity Analysis

If CCE threshold was too high/low, test different thresholds.

In [ ]:
# Test multiple thresholds
THRESHOLDS_TO_TEST = [1.0, 1.5, 2.0, 2.5, 3.0]

threshold_results = []

for threshold in THRESHOLDS_TO_TEST:
    print(f"\nTesting threshold: {threshold}")
    results = run_condition(DIFFICULT_QUESTIONS, 'cce_triggered', cce_threshold=threshold)
    
    accuracy = sum(1 for r in results if r.is_correct) / len(results)
    retrievals = sum(r.retrieval_count for r in results)
    
    threshold_results.append({
        'threshold': threshold,
        'accuracy': accuracy,
        'retrievals': retrievals,
    })
    
    print(f"  Accuracy: {accuracy:.1%}, Retrievals: {retrievals}")

df_thresholds = pd.DataFrame(threshold_results)
print("\n" + "="*50)
print("THRESHOLD SENSITIVITY ANALYSIS")
print("="*50)
print(df_thresholds.to_string(index=False))

In [ ]:
# Plot threshold sensitivity
fig, ax1 = plt.subplots(figsize=(10, 6))

ax1.set_xlabel('CCE Threshold')
ax1.set_ylabel('Accuracy', color='blue')
ax1.plot(df_thresholds['threshold'], df_thresholds['accuracy'], 'b-o', label='Accuracy')
ax1.tick_params(axis='y', labelcolor='blue')
ax1.axhline(y=no_ret_acc, color='red', linestyle='--', label='No Retrieval Baseline')

ax2 = ax1.twinx()
ax2.set_ylabel('Retrieval Count', color='green')
ax2.plot(df_thresholds['threshold'], df_thresholds['retrievals'], 'g-s', label='Retrievals')
ax2.tick_params(axis='y', labelcolor='green')

fig.legend(loc='upper right', bbox_to_anchor=(0.85, 0.85))
plt.title('CCE Threshold vs Accuracy and Retrieval Count')
plt.tight_layout()
plt.savefig('week13_threshold_sensitivity.png', dpi=150)
plt.show()